In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers openai-whisper --quiet
!pip install torch torchaudio --quiet
!pip install scikit-learn --quiet
!pip install nltk --quiet
!apt-get install -y ffmpeg --quiet

import os, json, numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

DATA        = '/content/drive/MyDrive/Data'
MODEL_SAVE  = '/content/drive/MyDrive/Models'
ARRAYS_PATH = '/content/drive/MyDrive/Arrays'
TEXT_ARRAYS = f'{ARRAYS_PATH}/text'
os.makedirs(MODEL_SAVE, exist_ok=True)

print(f"✅ PyTorch  : {torch.__version__}")
print(f"✅ GPU      : {torch.cuda.is_available()}")
print(f"✅ Device   : "
      f"{'cuda' if torch.cuda.is_available() else 'cpu'}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
✅ PyTorch  : 2.10.0+cu128
✅ GPU      : True
✅ Device   : cuda


In [ ]:
# ── Whisper ASR ───────────────────────────────────────
import whisper

print("Loading Whisper base model...")
whisper_model = whisper.load_model("base")
print(f"✅ Whisper loaded")

def transcribe_audio(audio_path):
    """Transcribe audio file to text"""
    try:
        result = whisper_model.transcribe(
            audio_path,
            language='en',
            fp16=torch.cuda.is_available())
        return {
            'text'    : result['text'].strip(),
            'segments': result['segments'],
            'language': result['language']
        }
    except Exception as e:
        print(f"Error: {e}")
        return {'text': '', 'segments': []}

print("✅ Whisper transcriber ready")
print("   Usage: transcribe_audio('path.wav')")

Loading Whisper base model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 103MiB/s]


✅ Whisper loaded
✅ Whisper transcriber ready
   Usage: transcribe_audio('path.wav')


In [ ]:
# ── Load saved RoBERTa model ──────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification)

ROBERTA_PATH = '/content/drive/MyDrive/final_roberta_model'

print("Loading RoBERTa model...")
roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification\
    .from_pretrained(ROBERTA_PATH)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')
roberta_model = roberta_model.to(device)
roberta_model.eval()

# Load emotion map
with open(f'{ROBERTA_PATH}/emotion_map.json') as f:
    emotion2id = json.load(f)
id2emotion = {v:k for k,v in emotion2id.items()}

print(f"✅ RoBERTa loaded")
print(f"   Emotions : {id2emotion}")

def predict_text_emotion(text):
    """Predict emotion from text"""
    inputs = roberta_tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding=True).to(device)

    with torch.no_grad():
        outputs = roberta_model(**inputs)
        probs   = torch.softmax(
            outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1)

    emotion = id2emotion[pred.item()]
    confidence = probs[0][pred.item()].item()

    return {
        'emotion'    : emotion,
        'confidence' : confidence,
        'all_probs'  : {
            id2emotion[i]: probs[0][i].item()
            for i in range(len(id2emotion))}
    }

# Test
test = predict_text_emotion(
    "I am very excited about this opportunity!")
print(f"\n✅ Test prediction:")
print(f"   Text     : I am very excited...")
print(f"   Emotion  : {test['emotion']}")
print(f"   Confidence: {test['confidence']:.3f}")

Loading RoBERTa model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ RoBERTa loaded
   Emotions : {0: 'angry', 1: 'anxious', 2: 'positive', 3: 'surprised'}

✅ Test prediction:
   Text     : I am very excited...
   Emotion  : positive
   Confidence: 1.000


In [ ]:
# ── Speech Quality Analyzer ───────────────────────────
import re
from collections import Counter

FILLER_WORDS = [
    'um', 'uh', 'like', 'so', 'you know',
    'basically', 'literally', 'actually',
    'i mean', 'right', 'okay so', 'well',
    'kind of', 'sort of', 'you see']

def analyze_speech_quality(text):
    """Analyze speech quality from transcript"""
    if not text:
        return {}

    words = text.lower().split()
    total_words = len(words)
    if total_words == 0:
        return {}

    # Filler word count
    filler_count = 0
    for filler in FILLER_WORDS:
        pattern = r'\b' + re.escape(filler) + r'\b'
        matches = re.findall(pattern, text.lower())
        filler_count += len(matches)

    filler_ratio = filler_count / total_words

    # Vocabulary richness (Type-Token Ratio)
    unique_words = set(words)
    ttr = len(unique_words) / total_words

    # Sentence count
    sentences = re.split(
        r'[.!?]+', text.strip())
    sentences = [s for s in sentences
                 if len(s.strip()) > 0]
    num_sentences = max(len(sentences), 1)

    # Average words per sentence
    avg_words_per_sent = total_words / \
        num_sentences

    # Clarity score (0-100)
    clarity = max(0, min(100,
        (1 - filler_ratio * 3) * 100))

    # Fluency score (0-100)
    fluency = min(100,
        ttr * 100 * 0.7 +
        min(avg_words_per_sent / 20, 1) * 30)

    return {
        'total_words'      : total_words,
        'unique_words'     : len(unique_words),
        'filler_count'     : filler_count,
        'filler_ratio'     : round(filler_ratio, 3),
        'vocabulary_richness': round(ttr, 3),
        'avg_words_per_sent': round(
            avg_words_per_sent, 1),
        'clarity_score'    : round(clarity, 1),
        'fluency_score'    : round(fluency, 1)
    }

# Test
test_text = """Um, I think like, you know,
    I have worked on many projects basically
    and I am very passionate about technology."""
result = analyze_speech_quality(test_text)
print("✅ Speech Quality Analysis:")
for k, v in result.items():
    print(f"   {k:25s}: {v}")

✅ Speech Quality Analysis:
   total_words              : 20
   unique_words             : 18
   filler_count             : 4
   filler_ratio             : 0.2
   vocabulary_richness      : 0.9
   avg_words_per_sent       : 20.0
   clarity_score            : 40.0
   fluency_score            : 93.0


In [ ]:
# ── Job Description Keyword Matcher ───────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
from nltk.corpus import stopwords

import pickle
import os

PIPELINE_SAVE = f'{MODEL_SAVE}/text_pipeline'
os.makedirs(PIPELINE_SAVE, exist_ok=True)

# Load Job Descriptions dataset
JD_PATH = f'{DATA}/job_title_des'
df_jd   = None

for f in os.listdir(JD_PATH):
    if f.endswith('.csv'):
        df_jd = pd.read_csv(
            os.path.join(JD_PATH, f))
        print(f"✅ JD loaded : {len(df_jd):,} rows")
        print(f"   Columns  : {df_jd.columns.tolist()}")
        print(df_jd.head(2))
        break

# Build TF-IDF on job descriptions
jd_col = None
for col in df_jd.columns:
    if 'desc' in col.lower() or \
       'job' in col.lower():
        jd_col = col
        break

if jd_col is None:
    jd_col = df_jd.columns[-1]

print(f"\nUsing column: {jd_col}")

# ── Fix JD vectorizer ─────────────────────────────────
# Use Job Description column not Job Title
jd_vectorizer_fixed = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english')

jd_matrix = jd_vectorizer_fixed.fit_transform(
    df_jd['Job Description'].fillna('').tolist())

# Save fixed vectorizer
with open(f'{PIPELINE_SAVE}/jd_vectorizer.pkl',
          'wb') as f:
    pickle.dump(jd_vectorizer_fixed, f)

jd_vectorizer = jd_vectorizer_fixed
print(f"✅ JD vectorizer fixed")
print(f"   Matrix : {jd_matrix.shape}")

✅ JD loaded : 2,277 rows
   Columns  : ['Unnamed: 0', 'Job Title', 'Job Description']
   Unnamed: 0          Job Title  \
0           0  Flutter Developer   
1           1   Django Developer   

                                     Job Description  
0  We are looking for hire experts flutter develo...  
1  PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...  

Using column: Job Title
✅ JD vectorizer fixed
   Matrix : (2277, 10000)


In [ ]:
# ── Keyword Match Function ────────────────────────────
# Interview-relevant skill keywords
SKILL_KEYWORDS = {
    'technical'   : [
        'python','java','sql','machine learning',
        'deep learning','api','algorithm',
        'data structure','cloud','docker',
        'tensorflow','pytorch','nlp','cv'],
    'leadership'  : [
        'led','managed','team','initiative',
        'ownership','mentored','coordinated',
        'supervised','directed','organized'],
    'problem_solving': [
        'solved','debugged','optimized',
        'improved','fixed','analyzed',
        'designed','implemented','resolved'],
    'communication': [
        'presented','explained','collaborated',
        'discussed','communicated','reported',
        'documented','trained','facilitated'],
    'star_method' : [
        'situation','task','action','result',
        'challenge','approach','outcome',
        'achieved','accomplished','delivered']
}

def compute_keyword_score(answer_text,
                          job_title=None):
    """Compute keyword relevance score"""
    if not answer_text:
        return {}

    answer_lower = answer_text.lower()
    scores = {}

    # Per category score
    for category, keywords in \
            SKILL_KEYWORDS.items():
        found = [kw for kw in keywords
                 if kw in answer_lower]
        score = min(100,
            len(found)/len(keywords)*100*2)
        scores[category] = {
            'score'    : round(score, 1),
            'found'    : found,
            'total_kw' : len(keywords)
        }

    # Overall keyword score
    total_score = np.mean([
        v['score'] for v in scores.values()])
    scores['overall'] = round(
        float(total_score), 1)

    # JD match score if job title provided
    if job_title and df_jd is not None:
        try:
            # Find matching JD
            title_col = df_jd.columns[1]
            jd_match  = df_jd[
                df_jd[title_col].str.contains(
                    job_title,
                    case=False,
                    na=False)]

            if len(jd_match) > 0:
                jd_text = jd_match[
                    jd_col].iloc[0]
                # Compute cosine similarity
                ans_vec = jd_vectorizer\
                    .transform([answer_text])
                jd_vec  = jd_vectorizer\
                    .transform([jd_text])
                sim = cosine_similarity(
                    ans_vec, jd_vec)[0][0]
                scores['jd_match'] = round(
                    float(sim * 100), 1)
            else:
                scores['jd_match'] = 0.0
        except:
            scores['jd_match'] = 0.0

    return scores

# Test
test_ans = """I led a team of 5 engineers
    to solve a critical performance issue.
    I analyzed the bottleneck, implemented
    an optimized algorithm using Python and
    achieved 40% improvement in speed."""

kw_result = compute_keyword_score(
    test_ans, job_title="Software Engineer")
print("✅ Keyword Score Test:")
for k, v in kw_result.items():
    print(f"   {k:15s}: {v}")

✅ Keyword Score Test:
   technical      : {'score': 28.6, 'found': ['python', 'algorithm'], 'total_kw': 14}
   leadership     : {'score': 40.0, 'found': ['led', 'team'], 'total_kw': 10}
   problem_solving: {'score': 66.7, 'found': ['optimized', 'analyzed', 'implemented'], 'total_kw': 9}
   communication  : {'score': 0.0, 'found': [], 'total_kw': 9}
   star_method    : {'score': 20.0, 'found': ['achieved'], 'total_kw': 10}
   overall        : 31.1
   jd_match       : 0.0


In [ ]:
# ── Answer Quality Scorer ─────────────────────────────
def score_answer_quality(text,
                         job_title=None):
    """
    Complete answer quality analysis
    Returns 0-100 score per dimension
    """
    if not text or len(text) < 10:
        return {
            'clarity'        : 0,
            'vocabulary'     : 0,
            'keyword_match'  : 0,
            'star_structure' : 0,
            'overall'        : 0
        }

    # Speech quality
    speech = analyze_speech_quality(text)

    # Keyword match
    kw = compute_keyword_score(
        text, job_title)

    # STAR method score
    star_score = kw.get(
        'star_method', {}).get('score', 0)

    # Answer length score (optimal 100-300 words)
    wc = speech.get('total_words', 0)
    if wc < 20:
        length_score = wc / 20 * 50
    elif wc <= 300:
        length_score = 50 + (wc-20)/280*50
    else:
        length_score = max(50,
            100 - (wc-300)/10)

    # Emotion score from RoBERTa
    emotion_result = predict_text_emotion(text)
    emotion = emotion_result['emotion']

    # Positive emotion → higher score
    emotion_score = {
        'positive' : 90,
        'surprised': 70,
        'anxious'  : 40,
        'angry'    : 20
    }.get(emotion, 50)

    # Combine scores
    clarity    = speech.get('clarity_score', 0)
    vocabulary = speech.get('fluency_score', 0)
    keyword    = kw.get('overall', 0)

    overall = (
        clarity    * 0.25 +
        vocabulary * 0.20 +
        keyword    * 0.25 +
        star_score * 0.15 +
        emotion_score * 0.15)

    return {
        'clarity'        : round(clarity, 1),
        'vocabulary'     : round(vocabulary, 1),
        'keyword_match'  : round(keyword, 1),
        'star_structure' : round(star_score, 1),
        'emotion'        : emotion,
        'emotion_score'  : emotion_score,
        'word_count'     : wc,
        'filler_ratio'   : speech.get(
            'filler_ratio', 0),
        'overall'        : round(overall, 1)
    }

# Test
print("✅ Answer Quality Test:")
test_result = score_answer_quality(
    test_ans,
    job_title="Software Engineer")
for k, v in test_result.items():
    print(f"   {k:20s}: {v}")

✅ Answer Quality Test:
   clarity             : 100
   vocabulary          : 86.9
   keyword_match       : 31.1
   star_structure      : 20.0
   emotion             : positive
   emotion_score       : 90
   word_count          : 29
   filler_ratio        : 0.0
   overall             : 66.7


In [ ]:
# ── Complete Text Pipeline ────────────────────────────
def analyze_interview_text(
        audio_path=None,
        text=None,
        job_title=None):
    """
    Complete pipeline:
    audio → transcript → emotion + quality
    OR
    text → emotion + quality
    """
    result = {}

    # Step 1 — Get transcript
    if audio_path:
        print("Transcribing audio...")
        transcript = transcribe_audio(
            audio_path)
        text = transcript['text']
        result['transcript'] = transcript
    else:
        result['transcript'] = {
            'text': text}

    if not text:
        return {'error': 'No text available'}

    result['text'] = text

    # Step 2 — Emotion prediction
    print("Predicting emotion...")
    emotion = predict_text_emotion(text)
    result['emotion'] = emotion

    # Step 3 — Answer quality
    print("Scoring answer quality...")
    quality = score_answer_quality(
        text, job_title)
    result['quality'] = quality

    # Step 4 — Speech metrics
    speech = analyze_speech_quality(text)
    result['speech_metrics'] = speech

    # Step 5 — Keywords
    keywords = compute_keyword_score(
        text, job_title)
    result['keywords'] = keywords

    return result

print("✅ Complete text pipeline ready!")

✅ Complete text pipeline ready!


In [ ]:
# ── Test complete pipeline ────────────────────────────
sample_answer = """
In my previous role as a software engineer,
I led a team of 5 developers to build a
real-time recommendation system. The situation
was that our platform was losing users due to
poor content discovery. My task was to design
and implement a solution within 3 months.
I analyzed user behavior data using Python
and machine learning algorithms, collaborated
with the product team, and delivered an
optimized system that improved engagement by
40 percent. The result was a significant
reduction in churn and positive feedback
from stakeholders.
"""

print("🎯 Running complete text pipeline...")
print("="*50)

result = analyze_interview_text(
    text=sample_answer,
    job_title="Software Engineer")

print(f"\n📝 TRANSCRIPT:")
print(f"   {result['text'][:100]}...")

print(f"\n😊 EMOTION ANALYSIS:")
print(f"   Emotion    : {result['emotion']['emotion']}")
print(f"   Confidence : "
      f"{result['emotion']['confidence']:.2f}")
print(f"   All probs  : ")
for e, p in result['emotion'][
        'all_probs'].items():
    print(f"     {e:10s}: {p:.3f}")

print(f"\n📊 ANSWER QUALITY:")
for k, v in result['quality'].items():
    print(f"   {k:20s}: {v}")

print(f"\n🎤 SPEECH METRICS:")
for k, v in result['speech_metrics'].items():
    print(f"   {k:25s}: {v}")

print(f"\n🔑 KEYWORD SCORES:")
for k, v in result['keywords'].items():
    if isinstance(v, dict):
        print(f"   {k:15s}: "
              f"{v['score']:.1f}% "
              f"(found: {v['found']})")
    else:
        print(f"   {k:15s}: {v}")

🎯 Running complete text pipeline...
Predicting emotion...
Scoring answer quality...

📝 TRANSCRIPT:
   
In my previous role as a software engineer,
I led a team of 5 developers to build a
real-time recom...

😊 EMOTION ANALYSIS:
   Emotion    : anxious
   Confidence : 0.92
   All probs  : 
     angry     : 0.030
     anxious   : 0.916
     positive  : 0.047
     surprised : 0.007

📊 ANSWER QUALITY:
   clarity             : 100
   vocabulary          : 81.6
   keyword_match       : 50.4
   star_structure      : 80.0
   emotion             : anxious
   emotion_score       : 40
   word_count          : 87
   filler_ratio        : 0.0
   overall             : 71.9

🎤 SPEECH METRICS:
   total_words              : 87
   unique_words             : 69
   filler_count             : 0
   filler_ratio             : 0.0
   vocabulary_richness      : 0.793
   avg_words_per_sent       : 17.4
   clarity_score            : 100
   fluency_score            : 81.6

🔑 KEYWORD SCORES:
   technical      : 42.

In [ ]:
# ── Save pipeline components ──────────────────────────
import pickle, json

PIPELINE_SAVE = f'{MODEL_SAVE}/text_pipeline'
os.makedirs(PIPELINE_SAVE, exist_ok=True)

# Save JD vectorizer
with open(f'{PIPELINE_SAVE}/jd_vectorizer.pkl',
          'wb') as f:
    pickle.dump(jd_vectorizer, f)

# Save keyword config
with open(f'{PIPELINE_SAVE}/skill_keywords.json',
          'w') as f:
    json.dump(SKILL_KEYWORDS, f, indent=2)

# Save pipeline metadata
pipeline_meta = {
    'whisper_model'   : 'base',
    'roberta_model'   : 'final_roberta_model',
    'emotion_map'     : id2emotion,
    'num_emotions'    : len(id2emotion),
    'jd_vectorizer'   : 'jd_vectorizer.pkl',
    'skill_categories': list(
        SKILL_KEYWORDS.keys()),
    'score_weights'   : {
        'clarity'    : 0.25,
        'vocabulary' : 0.20,
        'keywords'   : 0.25,
        'star_method': 0.15,
        'emotion'    : 0.15
    }
}

with open(f'{PIPELINE_SAVE}/pipeline_meta.json',
          'w') as f:
    json.dump(pipeline_meta, f, indent=2)

print("✅ Pipeline saved!")
print(f"   {PIPELINE_SAVE}/")
for f in os.listdir(PIPELINE_SAVE):
    size = os.path.getsize(
        f'{PIPELINE_SAVE}/{f}')
    print(f"   {f} — {size/1024:.1f} KB")

print(f"  Whisper ASR    : base model ✅")
print(f"  Text emotion   : RoBERTa 86% ✅")
print(f"  Speech quality : filler+TTR ✅")
print(f"  Keyword match  : TF-IDF+cos ✅")
print(f"  Answer quality : weighted score ✅")
print(f"  JD matching    : cosine sim ✅")